In [ ]:
!pip install grad-cam -q

import os, random, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
from collections import Counter
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from torchvision.utils import make_grid

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, precision_recall_curve,
    f1_score, recall_score, precision_score, average_precision_score
)
from sklearn.calibration import calibration_curve

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# Colors
NAVY  = "#0D1B2A"; BLUE = "#185FA5"; RED = "#A32D2D"
TEAL  = "#0F6E56"; AMBER = "#C47A1E"; PURPLE = "#534AB7"
GREEN = "#3B6D11"; GRAY = "#5F5E5A"; WHITE = "#FFFFFF"

plt.rcParams.update({
    'figure.facecolor': WHITE, 'axes.facecolor': '#FAFAF8',
    'axes.grid': True, 'grid.color': '#E8E8E8',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titlesize': 12, 'axes.labelsize': 10,
})

In [ ]:
import os
import pandas as pd

# ── FINAL DATASET: original Kaggle slices + ISLES-2022 + IXI-2022 ───────
# See build_final_dataset.py (project root) for the full pipeline.
#   - kaggle_slices : original Normal/Ischemic/Haemorrhagic 2D slices, harmonized
#   - isles2022     : 247 real ischemic-stroke patients (Ischemic only)
#   - ixi_normal    : 150 real healthy-control patients (Normal only)
# manifest.csv's `split` column is a patient-grouped train/val/test split,
# computed once at dataset-build time -- no single patient's images ever
# appear in more than one split (verified below).

CANDIDATE_ROOTS = [
    "/kaggle/input/datasets/karan2653raj/stroke-final-dataset/final_dataset",  # Kaggle (this project's uploaded dataset)
    "/kaggle/input/stroke-final-dataset/final_dataset",                        # Kaggle, alternate slug
    "/content/drive/MyDrive/stroke_prediction_jupyter_nb/dataset/final_dataset",  # Colab + Drive mount
    "/content/stroke_prediction_jupyter_nb/dataset/final_dataset",             # Colab, zip-upload
    "C:/stroke_prediction_jupyter_nb/dataset/final_dataset",                   # local Windows machine
    "./dataset/final_dataset",                                                  # notebook launched from project root
]
DATA_ROOT = next((p for p in CANDIDATE_ROOTS if os.path.exists(os.path.join(p, "manifest.csv"))), None)
assert DATA_ROOT is not None, (
    "Could not find final_dataset/manifest.csv under any candidate root -- "
    "if the dataset was re-uploaded under a different Kaggle slug/path, add "
    "it to CANDIDATE_ROOTS above."
)
print(f"DATA_ROOT = {DATA_ROOT}")

MANIFEST_PATH = os.path.join(DATA_ROOT, "manifest.csv")
assert os.path.exists(MANIFEST_PATH), f"Manifest not found: {MANIFEST_PATH}"

CLASS_TO_LABEL = {"Normal": 0, "Ischemic": 1, "Haemorrhagic": 2}
CLASS_NAMES    = ["Normal", "Ischemic", "Haemorrhagic"]
NUM_CLASSES    = len(CLASS_NAMES)

CLASS_TO_RISK = {
    "Normal": "No Stroke",
    "Ischemic": "Ischemic Stroke",
    "Haemorrhagic": "Haemorrhagic Stroke",
}
RISK_COLORS = {
    "No Stroke": GREEN,
    "Ischemic Stroke": AMBER,
    "Haemorrhagic Stroke": RED,
}

df = pd.read_csv(MANIFEST_PATH)


def fix_path(p):
    """manifest.csv stores paths like 'dataset/final_dataset/<Class>/<file>.png'
    -- strip whatever prefix precedes 'final_dataset/' and re-anchor at
    DATA_ROOT, since different platforms mount/extract it at different
    nesting depths (Kaggle drops it straight at the dataset root, Colab/
    local keep the original dataset/final_dataset/ nesting)."""
    p = str(p).replace("\\", "/")
    if p.startswith("dataset/final_dataset/"):
        p = p[len("dataset/final_dataset/"):]
    elif p.startswith("final_dataset/"):
        p = p[len("final_dataset/"):]
    return os.path.join(DATA_ROOT, p)


df["path"] = df["path"].apply(fix_path)
df["risk"] = df["mri_class"].map(CLASS_TO_RISK)

print("\nChecking image paths:")
print(df["path"].iloc[0])
print("Exists:", os.path.exists(df["path"].iloc[0]))

for cls in CLASS_NAMES:
    sub = df[df["mri_class"] == cls]
    print(f"{cls:15s}: {len(sub):4d} images  |  {sub['patient_id'].nunique():3d} patients  "
          f"→  label={CLASS_TO_LABEL[cls]}  ({CLASS_TO_RISK[cls]})")


In [ ]:
print(f"Total images   : {len(df)}")
print(f"Total patients : {df['patient_id'].nunique()}")
print("\n3-class label distribution (images):")
print(df["label"].value_counts().sort_index())

print("\nImages by class x source:")
print(df.groupby(["mri_class", "source"]).size())

print("\nPatients by class x source:")
print(df.groupby(["mri_class", "source"])["patient_id"].nunique())

print(
    "\n[OK] Normal now has 156 patients (6 original + 150 from IXI-2022 "
    "healthy controls) and Ischemic has 252 (5 original + 247 from "
    "ISLES-2022) -- both are well past the point of being critically "
    "patient-thin.\n"
    "[!] Haemorrhagic is still the bottleneck: 5 total patients, unchanged. "
    "Neither IXI (healthy-only) nor ISLES (ischemic-only) could help this "
    "class. Its test-set metrics remain statistically thin (small patient "
    "count) -- treat them as a rough signal, not a precise estimate, until "
    "a real hemorrhage-MRI source is found."
)


In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(18, 9))
fig.suptitle("Fig 2 — Sample MRI Images by Class & Risk Tier",
             fontsize=14, fontweight='bold', color=NAVY)

for row, cls in enumerate(["Normal", "Ischemic", "Haemorrhagic"]):
    subset = df[df["mri_class"] == cls].sample(6, random_state=SEED)
    risk   = CLASS_TO_RISK[cls]
    color  = RISK_COLORS[risk]
    for col, (_, row_data) in enumerate(subset.iterrows()):
        img = Image.open(row_data["path"]).convert("RGB")
        axes[row][col].imshow(img, cmap='gray')
        axes[row][col].axis('off')
        if col == 0:
            axes[row][col].set_ylabel(f"{cls}\n({risk})",
                                       fontsize=10, fontweight='bold', color=color,
                                       rotation=0, labelpad=80, va='center')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Fig 1 — Dataset Overview", fontsize=14, fontweight='bold', color=NAVY, y=1.02)

# MRI class counts
cls_counts = df["mri_class"].value_counts().reindex(CLASS_NAMES)
colors_cls = [GREEN, AMBER, RED]
bars = axes[0].bar(cls_counts.index, cls_counts.values, color=colors_cls,
                   edgecolor='white', width=0.5)
axes[0].set_title("MRI Class Distribution", fontweight='bold')
axes[0].set_ylabel("Image Count")
for bar, val in zip(bars, cls_counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+3,
                 str(val), ha='center', fontweight='bold', fontsize=12)

# Stroke type pie — now uses the 3 actual class names
risk_counts = df["mri_class"].value_counts().reindex(CLASS_NAMES)
axes[1].pie(risk_counts.values, labels=CLASS_NAMES, colors=[GREEN, AMBER, RED],
            autopct='%1.1f%%', startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title("Stroke Type Distribution", fontweight='bold')

# 3-class label bar (replaces old binary bar)
label_counts = df["label"].value_counts().sort_index()
axes[2].bar(CLASS_NAMES, label_counts.values, color=[GREEN, AMBER, RED],
            edgecolor='white', width=0.5)
axes[2].set_title("3-Class Label Distribution", fontweight='bold')
axes[2].set_ylabel("Image Count")
for i, val in enumerate(label_counts.values):
    axes[2].text(i, val+3, str(val), ha='center', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Fig 3 — Pixel Intensity Distributions by Class",
             fontsize=14, fontweight='bold', color=NAVY)

cls_colors = {"Normal": GREEN, "Ischemic": AMBER, "Haemorrhagic": RED}

for cls, col in cls_colors.items():
    sample = df[df["mri_class"]==cls].sample(min(30, len(df[df["mri_class"]==cls])), random_state=SEED)
    pixels = []
    for _, row_data in sample.iterrows():
        img = np.array(Image.open(row_data["path"]).convert("L"))
        pixels.extend(img.flatten().tolist())

    axes[0].hist(pixels, bins=60, alpha=0.5, color=col, label=cls, density=True)
    # KDE manually
    from scipy.stats import gaussian_kde
    kde = gaussian_kde(pixels[:5000])
    x   = np.linspace(0, 255, 300)
    axes[0].plot(x, kde(x), color=col, linewidth=2)

axes[0].set_xlabel("Pixel Intensity (0=black, 255=white)")
axes[0].set_ylabel("Density")
axes[0].set_title("Overlaid Intensity Distributions", fontweight='bold')
axes[0].legend()

# Mean intensity per class
mean_intensities = []
for cls in ["Normal", "Ischemic", "Haemorrhagic"]:
    sample = df[df["mri_class"]==cls].sample(min(50, len(df[df["mri_class"]==cls])), random_state=SEED)
    means  = [np.array(Image.open(r["path"]).convert("L")).mean() for _, r in sample.iterrows()]
    mean_intensities.append(means)

bp = axes[1].boxplot(mean_intensities, patch_artist=True,
                      labels=["Normal","Ischemic","Haemorrhagic"],
                      medianprops=dict(color='white', linewidth=2))
for patch, col in zip(bp['boxes'], [GREEN, AMBER, RED]):
    patch.set_facecolor(col + 'AA')
axes[1].set_ylabel("Mean Pixel Intensity per Image")
axes[1].set_title("Mean Brightness by Class\n(Haemorrhagic tends brighter due to bleeding)", fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Patient-grouped split -- computed once at dataset-build time and stored
# in manifest.csv's `split` column (see build_final_dataset.py::assign_splits).
# This guarantees no single patient's images appear in more than one of
# train/val/test. The old approach here was:
#   train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["label"])
# which stratified on label at IMAGE level with no notion of patient identity
# -- given many near-duplicate slices per patient, that let the same patient
# leak across splits and inflated the old val/test metrics.

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df   = df[df["split"] == "val"].reset_index(drop=True)
test_df  = df[df["split"] == "test"].reset_index(drop=True)

for split, d in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    counts = d["label"].value_counts().sort_index()
    print(f"{split:5s}: {len(d):4d} images | {d['patient_id'].nunique():3d} patients | " +
          " | ".join(f"{CLASS_NAMES[i]}={counts.get(i, 0)}" for i in range(3)))

# Leakage sanity check -- every patient must fall in exactly one split
patient_splits = df.groupby("patient_id")["split"].nunique()
assert (patient_splits == 1).all(), "Patient leakage detected across splits!"
print("\nPatient-leakage check passed: every patient appears in exactly one split.")


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class MRIDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        img   = Image.open(row["path"]).convert("RGB")
        label = torch.tensor(row["label"], dtype=torch.long)
        if self.transform:
            img = self.transform(img)
        return img, label, row["mri_class"], row["risk"]

train_dataset = MRIDataset(train_df, train_transforms)
val_dataset   = MRIDataset(val_df,   val_transforms)
test_dataset  = MRIDataset(test_df,  val_transforms)

# Weighted sampler so minority class (Stroke) gets equal sampling per batch
class_counts  = train_df["label"].value_counts().sort_index().values
weights       = 1.0 / class_counts
sample_weights = [weights[l] for l in train_df["label"].values]
sampler       = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=16, sampler=sampler, num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=16, shuffle=False,  num_workers=2)

print(f"Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

In [ ]:
sample_img, _, cls_name, risk_name = train_dataset[0]

# Denormalize for display
def denorm(tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3,1,1)
    std  = torch.tensor(IMAGENET_STD).view(3,1,1)
    return torch.clamp(tensor * std + mean, 0, 1)

fig, axes = plt.subplots(2, 6, figsize=(18, 6))
fig.suptitle("Fig 4 — Augmentation Examples (same image, 6 different augmentations)",
             fontsize=13, fontweight='bold', color=NAVY)

orig = Image.open(train_df.iloc[0]["path"]).convert("RGB")
axes[0][0].imshow(orig); axes[0][0].set_title("Original", fontweight='bold')
axes[0][0].axis('off')

for i in range(1, 6):
    aug, _, _, _ = train_dataset[0]
    axes[0][i].imshow(denorm(aug).permute(1,2,0).numpy())
    axes[0][i].set_title(f"Aug {i}"); axes[0][i].axis('off')

orig2 = Image.open(train_df.iloc[5]["path"]).convert("RGB")
axes[1][0].imshow(orig2); axes[1][0].set_title("Original", fontweight='bold')
axes[1][0].axis('off')

for i in range(1, 6):
    aug2, _, _, _ = train_dataset[5]
    axes[1][i].imshow(denorm(aug2).permute(1,2,0).numpy())
    axes[1][i].set_title(f"Aug {i}"); axes[1][i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
def build_resnet18_3class():
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    # Freeze backbone — only train head initially
    for param in model.parameters():
        param.requires_grad = False
    # Replace final FC: 512 → 3 (Normal | Ischemic | Haemorrhagic)
    model.fc = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Dropout(p=0.2),
        nn.Linear(256, NUM_CLASSES)   # ← 3 outputs
    )
    return model

model = build_resnet18_3class().to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,}  (frozen backbone)")
print(f"Output classes  : {NUM_CLASSES} → Normal / Ischemic / Haemorrhagic")


In [ ]:
# Per-class counts
counts = train_df["label"].value_counts().sort_index().values   # [n_normal, n_isch, n_haem]
total  = counts.sum()
# Inverse-frequency weights
class_weights = torch.tensor(total / (NUM_CLASSES * counts), dtype=torch.float).to(DEVICE)
print(f"Class counts  : {dict(zip(CLASS_NAMES, counts))}")
print(f"Loss weights  : {dict(zip(CLASS_NAMES, class_weights.cpu().numpy().round(3)))}")

criterion = nn.CrossEntropyLoss(weight=class_weights)

# Phase 1: only train the new head
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                       lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
                                                  patience=3, factor=0.5)


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels, _, _ in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        correct      += (outputs.argmax(1) == labels).sum().item()
        total        += imgs.size(0)
    return running_loss/total, correct/total

def evaluate(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_probs_3, all_labels = [], []
    with torch.no_grad():
        for imgs, labels, _, _ in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs      = model(imgs)
            loss         = criterion(outputs, labels)
            probs_3      = torch.softmax(outputs, dim=1)   # (B, 3)
            running_loss += loss.item() * imgs.size(0)
            correct      += (outputs.argmax(1) == labels).sum().item()
            total        += imgs.size(0)
            all_probs_3.extend(probs_3.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_probs_3 = np.array(all_probs_3)
    all_labels  = np.array(all_labels)
    # Macro-average AUC for 3-class
    try:
        auc = roc_auc_score(all_labels, all_probs_3, multi_class='ovr', average='macro')
    except Exception:
        auc = 0.5
    return running_loss/total, correct/total, auc, all_probs_3, all_labels

history = {"train_loss":[], "val_loss":[], "train_acc":[], "val_acc":[], "val_auc":[]}

# ── Phase 1: Train head only (10 epochs) ──────────────────────────────────────
print("Phase 1 — Training head only (backbone frozen) | 3-class: Normal / Ischemic / Haemorrhagic")
print(f"{'Epoch':>6} {'Train Loss':>11} {'Train Acc':>10} {'Val Loss':>10} {'Val Acc':>9} {'Val AUC':>9} {'LR':>10}")
print("─"*70)

PHASE1_EPOCHS = 10
best_val_loss = float('inf')
best_state    = None

for epoch in range(1, PHASE1_EPOCHS+1):
    t0 = time.time()
    tr_loss, tr_acc               = train_one_epoch(model, train_loader, criterion, optimizer)
    vl_loss, vl_acc, vl_auc, _, _ = evaluate(model, val_loader, criterion)
    scheduler.step(vl_loss)
    history["train_loss"].append(tr_loss); history["val_loss"].append(vl_loss)
    history["train_acc"].append(tr_acc);   history["val_acc"].append(vl_acc)
    history["val_auc"].append(vl_auc)
    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        best_state    = {k: v.clone() for k, v in model.state_dict().items()}
    lr = optimizer.param_groups[0]['lr']
    print(f"{epoch:>6} {tr_loss:>11.4f} {tr_acc:>10.4f} {vl_loss:>10.4f} {vl_acc:>9.4f} {vl_auc:>9.4f} {lr:>10.6f}  ({time.time()-t0:.1f}s)")

# ── Phase 2: Unfreeze all, fine-tune ──────────────────────────────────────────
print("\nPhase 2 — Fine-tuning entire network (all layers unfrozen)")
for param in model.parameters():
    param.requires_grad = True

optimizer2 = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler2 = optim.lr_scheduler.ReduceLROnPlateau(optimizer2, mode='min',
                                                   patience=3, factor=0.5)

PHASE2_EPOCHS       = 15
early_stop_patience = 5
no_improve          = 0

print(f"{'Epoch':>6} {'Train Loss':>11} {'Train Acc':>10} {'Val Loss':>10} {'Val Acc':>9} {'Val AUC':>9} {'LR':>10}")
print("─"*70)

for epoch in range(PHASE1_EPOCHS+1, PHASE1_EPOCHS+PHASE2_EPOCHS+1):
    t0 = time.time()
    tr_loss, tr_acc               = train_one_epoch(model, train_loader, criterion, optimizer2)
    vl_loss, vl_acc, vl_auc, _, _ = evaluate(model, val_loader, criterion)
    scheduler2.step(vl_loss)
    history["train_loss"].append(tr_loss); history["val_loss"].append(vl_loss)
    history["train_acc"].append(tr_acc);   history["val_acc"].append(vl_acc)
    history["val_auc"].append(vl_auc)
    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        best_state    = {k: v.clone() for k, v in model.state_dict().items()}
        no_improve    = 0
    else:
        no_improve += 1
    lr = optimizer2.param_groups[0]['lr']
    print(f"{epoch:>6} {tr_loss:>11.4f} {tr_acc:>10.4f} {vl_loss:>10.4f} {vl_acc:>9.4f} {vl_auc:>9.4f} {lr:>10.6f}  ({time.time()-t0:.1f}s)")
    if no_improve >= early_stop_patience:
        print(f"\nEarly stopping at epoch {epoch} — val loss unchanged for {early_stop_patience} epochs")
        break

model.load_state_dict(best_state)
print("\nBest model restored.")


In [ ]:
epochs_ran = len(history["train_loss"])
x          = range(1, epochs_ran+1)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("Fig 5 — Training Curves (Phase 1 = frozen | Phase 2 = fine-tune)",
             fontsize=14, fontweight='bold', color=NAVY)

axes[0].plot(x, history["train_loss"], color=BLUE, linewidth=2, label="Train Loss")
axes[0].plot(x, history["val_loss"],   color=RED,  linewidth=2, label="Val Loss")
axes[0].axvline(PHASE1_EPOCHS, color=GRAY, linestyle='--', linewidth=1.2, label="Phase switch")
axes[0].set_title("Loss", fontweight='bold'); axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(x, history["train_acc"], color=BLUE, linewidth=2, label="Train Acc")
axes[1].plot(x, history["val_acc"],   color=RED,  linewidth=2, label="Val Acc")
axes[1].axvline(PHASE1_EPOCHS, color=GRAY, linestyle='--', linewidth=1.2, label="Phase switch")
axes[1].set_title("Accuracy", fontweight='bold'); axes[1].set_xlabel("Epoch")
axes[1].set_ylim(0, 1.05); axes[1].legend()

axes[2].plot(x, history["val_auc"], color=TEAL, linewidth=2, marker='o', markersize=3)
axes[2].axvline(PHASE1_EPOCHS, color=GRAY, linestyle='--', linewidth=1.2, label="Phase switch")
axes[2].axhline(0.85, color=RED, linestyle='--', linewidth=1.2, label="Target AUC 0.85")
axes[2].set_title("Val AUC-ROC", fontweight='bold'); axes[2].set_xlabel("Epoch")
axes[2].set_ylim(0.5, 1.05); axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
model.eval()
all_probs_3, all_labels, all_preds = [], [], []
all_mri_class = []

with torch.no_grad():
    for imgs, labels, mri_classes, risks in test_loader:
        imgs    = imgs.to(DEVICE)
        outputs = model(imgs)
        probs_3 = torch.softmax(outputs, dim=1)   # (B, 3)
        preds   = outputs.argmax(1)
        all_probs_3.extend(probs_3.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_preds.extend(preds.cpu().numpy())
        all_mri_class.extend(mri_classes)

all_probs_3  = np.array(all_probs_3)   # shape (N, 3)
all_labels   = np.array(all_labels)    # int 0/1/2
all_preds    = np.array(all_preds)     # int 0/1/2

# Human-readable predictions
pred_class_names = np.array([CLASS_NAMES[p] for p in all_preds])
true_class_names = np.array(all_mri_class)

print("Predicted class distribution:")
print(pd.Series(pred_class_names).value_counts())
print("\nTrue class distribution:")
print(pd.Series(true_class_names).value_counts())


In [ ]:
from matplotlib.colors import LinearSegmentedColormap

cm = confusion_matrix(true_class_names, pred_class_names, labels=CLASS_NAMES)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Fig 6 — 3-Class Confusion Matrix: Normal / Ischemic / Haemorrhagic",
             fontsize=14, fontweight='bold', color=NAVY)

cmap_cm = LinearSegmentedColormap.from_list('cm', ['#F0F8FF', '#185FA5'])
CLASS_COLORS_ROW = [GREEN, AMBER, RED]

sns.heatmap(cm, annot=False, cmap=cmap_cm, ax=axes[0],
            linewidths=1.5, linecolor='white', cbar=True,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
for i in range(3):
    for j in range(3):
        val = cm[i, j]; pct = cm_pct[i, j]
        color = 'white' if val > cm.max()*0.5 else NAVY
        axes[0].text(j+0.5, i+0.38, f'{val}',
                     ha='center', fontsize=16, fontweight='bold', color=color)
        axes[0].text(j+0.5, i+0.65, f'({pct:.1f}%)',
                     ha='center', fontsize=9, color=color)
axes[0].set_xlabel("Predicted Class", fontweight='bold')
axes[0].set_ylabel("True Class", fontweight='bold')
axes[0].set_title("Count + Row %", fontweight='bold')

sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='RdYlGn', ax=axes[1],
            linewidths=1.5, linecolor='white', cbar=True, vmin=0, vmax=100,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
axes[1].set_xlabel("Predicted Class", fontweight='bold')
axes[1].set_ylabel("True Class", fontweight='bold')
axes[1].set_title("Row-Normalized (Recall per Class)", fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
print("="*60)
print("  3-CLASS STROKE CLASSIFICATION REPORT")
print("="*60)
print(classification_report(true_class_names, pred_class_names,
                                     labels=CLASS_NAMES, target_names=CLASS_NAMES))

# Macro AUC
macro_auc = roc_auc_score(all_labels, all_probs_3, multi_class='ovr', average='macro')
print(f"Macro AUC-ROC (OvR) : {macro_auc:.4f}")

# Per-class AUC
for i, cls in enumerate(CLASS_NAMES):
    binary_labels = (all_labels == i).astype(int)
    auc_i = roc_auc_score(binary_labels, all_probs_3[:, i])
    print(f"  AUC ({cls:15s}): {auc_i:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("Fig 8 — Per-Class ROC Curves (One-vs-Rest)",
             fontsize=14, fontweight='bold', color=NAVY)

cls_colors_list = [GREEN, AMBER, RED]
for i, (cls, col) in enumerate(zip(CLASS_NAMES, cls_colors_list)):
    binary_labels = (all_labels == i).astype(int)
    fpr, tpr, _   = roc_curve(binary_labels, all_probs_3[:, i])
    auc_i         = roc_auc_score(binary_labels, all_probs_3[:, i])
    axes[i].plot(fpr, tpr, color=col, linewidth=2.5,
                 label=f"{cls} vs Rest\nAUC={auc_i:.3f}")
    axes[i].plot([0,1],[0,1], '--', color='#AAAAAA', linewidth=1)
    axes[i].fill_between(fpr, tpr, alpha=0.1, color=col)
    axes[i].set_xlabel("False Positive Rate")
    axes[i].set_ylabel("True Positive Rate")
    axes[i].set_title(f"{cls} (AUC={auc_i:.3f})", fontweight='bold',
                       color=col)
    axes[i].legend()

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("Fig 9 — Predicted Class Probability Distributions",
             fontsize=14, fontweight='bold', color=NAVY)

cls_colors_list = [GREEN, AMBER, RED]
for i, (cls_plot, col_plot) in enumerate(zip(CLASS_NAMES, cls_colors_list)):
    for j, (cls_true, col_true) in enumerate(zip(CLASS_NAMES, [GREEN, AMBER, RED])):
        mask  = all_labels == j
        probs = all_probs_3[mask, i]
        axes[i].hist(probs, bins=25, alpha=0.5, color=col_true,
                     label=f"True: {cls_true}", density=True)
    axes[i].set_xlabel(f"P({cls_plot})")
    axes[i].set_ylabel("Density")
    axes[i].set_title(f"Probability of {cls_plot}\nby true class", fontweight='bold')
    axes[i].legend(fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
# Confidence = max softmax probability
confidence = all_probs_3.max(axis=1)
correct_mask = (all_preds == all_labels)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Fig 10 — Model Confidence Analysis",
             fontsize=13, fontweight='bold', color=NAVY)

axes[0].hist(confidence[correct_mask],  bins=25, alpha=0.7, color=GREEN,
             label=f"Correct ({correct_mask.sum()})", density=True)
axes[0].hist(confidence[~correct_mask], bins=25, alpha=0.7, color=RED,
             label=f"Wrong ({(~correct_mask).sum()})", density=True)
axes[0].set_xlabel("Max Softmax Confidence")
axes[0].set_ylabel("Density")
axes[0].set_title("Confidence: Correct vs Wrong", fontweight='bold')
axes[0].legend()

# Per-class accuracy bar
per_class_acc = []
for i, cls in enumerate(CLASS_NAMES):
    mask_i = all_labels == i
    acc_i  = (all_preds[mask_i] == i).mean() if mask_i.sum() > 0 else 0
    per_class_acc.append(acc_i)
bars = axes[1].bar(CLASS_NAMES, per_class_acc, color=cls_colors_list, edgecolor='white', width=0.5)
axes[1].set_ylim(0, 1.1)
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Per-Class Accuracy", fontweight='bold')
for bar, val in zip(bars, per_class_acc):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
                 f"{val:.2%}", ha='center', fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.calibration import calibration_curve

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Fig 11 — Per-Class Calibration Curves",
             fontsize=13, fontweight='bold', color=NAVY)

cls_colors_list = [GREEN, AMBER, RED]
for i, (cls, col) in enumerate(zip(CLASS_NAMES, cls_colors_list)):
    binary_labels = (all_labels == i).astype(int)
    prob_true, prob_pred = calibration_curve(binary_labels, all_probs_3[:, i],
                                              n_bins=8, strategy='uniform')
    axes[i].plot([0,1],[0,1], 'k--', linewidth=1.5, label="Perfect")
    axes[i].plot(prob_pred, prob_true, color=col, marker='o', linewidth=2.5,
                 markersize=8, label=f"{cls}")
    axes[i].fill_between(prob_pred, prob_true, prob_pred, alpha=0.1, color=col)
    axes[i].set_xlabel(f"Mean P({cls})")
    axes[i].set_ylabel("Actual Fraction")
    axes[i].set_title(f"{cls} Calibration", fontweight='bold', color=col)
    axes[i].legend(); axes[i].set_xlim(0,1); axes[i].set_ylim(0,1)

plt.tight_layout()
plt.show()


In [ ]:
sample_idx = np.random.choice(len(test_df), size=12, replace=False)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.suptitle("Fig 12 — Individual Patient Diagnosis Cards\n"
             "Normal (green) | Ischemic (amber) | Haemorrhagic (red)",
             fontsize=13, fontweight='bold', color=NAVY)
axes = axes.flatten()

for i, idx in enumerate(sample_idx):
    row      = test_df.iloc[idx]
    img      = Image.open(row["path"]).convert("RGB")
    probs_i  = all_probs_3[idx]           # (3,)
    pred_cls = CLASS_NAMES[all_preds[idx]]
    true_cls = row["mri_class"]
    conf     = probs_i.max()
    correct  = pred_cls == true_cls
    col      = [GREEN, AMBER, RED][all_preds[idx]]

    axes[i].imshow(img)
    axes[i].axis('off')
    border_col = GREEN if correct else RED
    for spine in axes[i].spines.values():
        spine.set_visible(True)
        spine.set_edgecolor(border_col)
        spine.set_linewidth(3)

    prob_str = f"N:{probs_i[0]:.2f} I:{probs_i[1]:.2f} H:{probs_i[2]:.2f}"
    axes[i].set_title(
        f"Pred: {pred_cls}\nTrue: {true_cls}\n{prob_str}\n{'✓ Correct' if correct else '✗ Wrong'}",
        fontsize=7.5, fontweight='bold',
        color=GREEN if correct else RED,
        pad=4
    )

plt.tight_layout()
plt.show()


In [ ]:
# Grad-CAM on layer4 of ResNet-18 — now with 3 target classes
target_layer = [model.layer4[-1]]
cam          = GradCAM(model=model, target_layers=target_layer)

def get_gradcam_3class(img_path, target_class_idx):
    """target_class_idx: 0=Normal, 1=Ischemic, 2=Haemorrhagic"""
    img_pil = Image.open(img_path).convert("RGB")
    img_np  = np.array(img_pil.resize((224, 224))) / 255.0

    tensor  = val_transforms(img_pil).unsqueeze(0).to(DEVICE)
    targets = [ClassifierOutputTarget(target_class_idx)]
    grayscale_cam = cam(input_tensor=tensor, targets=targets)[0]

    rgb_img   = img_np.astype(np.float32)
    cam_image = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)[0].cpu().numpy()  # (3,)
    pred_cls = CLASS_NAMES[probs.argmax()]
    return img_np, grayscale_cam, cam_image, probs, pred_cls

print("Grad-CAM ready — 3 target classes: Normal / Ischemic / Haemorrhagic")


In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
fig.suptitle("Fig 14 — Grad-CAM: Model Focus per Stroke Type\n"
             "Original | Heatmap | Overlay + 2 more examples",
             fontsize=14, fontweight='bold', color=NAVY, y=1.01)

cls_target_idx = {"Normal": 0, "Ischemic": 1, "Haemorrhagic": 2}
cls_col_map    = {"Normal": GREEN, "Ischemic": AMBER, "Haemorrhagic": RED}

for row_i, cls in enumerate(CLASS_NAMES):
    subset = test_df[test_df["mri_class"] == cls]
    if len(subset) == 0:
        continue
    target_idx = cls_target_idx[cls]
    col_cls    = cls_col_map[cls]

    # First example: original + heatmap + overlay
    sample_row = subset.iloc[0]
    orig, heatmap, overlay, probs, pred_cls = get_gradcam_3class(sample_row["path"], target_idx)

    axes[row_i][0].imshow(orig);             axes[row_i][0].axis('off')
    axes[row_i][0].set_title("Original", fontweight='bold')

    axes[row_i][1].imshow(heatmap, cmap='jet'); axes[row_i][1].axis('off')
    axes[row_i][1].set_title("Grad-CAM", fontweight='bold')

    axes[row_i][2].imshow(overlay);         axes[row_i][2].axis('off')
    axes[row_i][2].set_title(
        f"Pred: {pred_cls}\nN:{probs[0]:.2f} I:{probs[1]:.2f} H:{probs[2]:.2f}",
        fontweight='bold', color=cls_col_map.get(pred_cls, NAVY), fontsize=8)

    # 2 more overlay examples
    for col_i, (_, r) in enumerate(subset.iloc[1:3].iterrows()):
        _, _, overlay2, probs2, pred2 = get_gradcam_3class(r["path"], target_idx)
        axes[row_i][col_i+3].imshow(overlay2); axes[row_i][col_i+3].axis('off')
        axes[row_i][col_i+3].set_title(
            f"Pred: {pred2}\nN:{probs2[0]:.2f} I:{probs2[1]:.2f} H:{probs2[2]:.2f}",
            fontweight='bold', color=cls_col_map.get(pred2, NAVY), fontsize=8)

    axes[row_i][0].set_ylabel(f"{cls}", fontsize=11, fontweight='bold',
                               color=col_cls, rotation=0, labelpad=80, va='center')

plt.tight_layout()
plt.show()


In [ ]:
# Show worst confusions: Ischemic predicted as Haemorrhagic and vice versa
# (the clinically critical errors)
print("Critical misclassifications (stroke-type confusion):")
for true_i, pred_i in [(1,2), (2,1), (0,1), (0,2)]:
    t_name = CLASS_NAMES[true_i]; p_name = CLASS_NAMES[pred_i]
    idxs = [i for i in range(len(test_df))
            if all_labels[i]==true_i and all_preds[i]==pred_i]
    print(f"  True={t_name}, Pred={p_name}: {len(idxs)} cases")

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle("Fig 15 — Critical Misclassifications\n"
             "Top: Ischemic predicted as Haemorrhagic | Bottom: Haemorrhagic predicted as Ischemic",
             fontsize=13, fontweight='bold', color=NAVY)

for row_i, (true_i, pred_i) in enumerate([(1, 2), (2, 1)]):
    idxs = [i for i in range(len(test_df))
            if all_labels[i]==true_i and all_preds[i]==pred_i]
    for col_i in range(4):
        ax = axes[row_i][col_i]
        if col_i < len(idxs):
            idx = idxs[col_i]
            row_data = test_df.iloc[idx]
            _, _, overlay, probs, pred_cls = get_gradcam_3class(
                row_data["path"], pred_i)
            ax.imshow(overlay); ax.axis('off')
            ax.set_title(
                f"True: {CLASS_NAMES[true_i]}\nPred: {CLASS_NAMES[pred_i]}\n"
                f"N:{probs[0]:.2f} I:{probs[1]:.2f} H:{probs[2]:.2f}",
                fontsize=8, fontweight='bold', color=RED)
        else:
            ax.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.gridspec as gridspec

macro_auc = roc_auc_score(all_labels, all_probs_3, multi_class='ovr', average='macro')
overall_acc = (all_preds == all_labels).mean()
macro_f1    = f1_score(all_labels, all_preds, average='macro')
macro_recall = recall_score(all_labels, all_preds, average='macro')

fig = plt.figure(figsize=(20, 14))
fig.patch.set_facecolor(NAVY)
fig.suptitle("BRAIN STROKE MRI — 3-CLASS DIAGNOSIS DASHBOARD\n"
             "Normal  |  Ischemic Stroke  |  Haemorrhagic Stroke",
             fontsize=16, fontweight='bold', color=WHITE, y=0.98)

gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.5, wspace=0.35,
                        top=0.92, bottom=0.06, left=0.06, right=0.96)

# Metric cards
card_vals = [
    (f"{macro_auc:.3f}",    "Macro AUC-ROC",   BLUE),
    (f"{overall_acc:.3f}",  "Overall Accuracy", TEAL),
    (f"{macro_recall:.3f}", "Macro Recall",     RED),
    (f"{macro_f1:.3f}",     "Macro F1",         PURPLE),
]
for j, (val, lbl, col) in enumerate(card_vals):
    ax = fig.add_subplot(gs[0, j])
    ax.set_facecolor(col)
    for sp in ax.spines.values(): sp.set_visible(False)
    ax.set_xticks([]); ax.set_yticks([])
    ax.text(0.5, 0.6, val, transform=ax.transAxes, ha='center',
            fontsize=24, fontweight='bold', color=WHITE)
    ax.text(0.5, 0.2, lbl, transform=ax.transAxes, ha='center',
            fontsize=11, color='white', alpha=0.85)

# Per-class ROC
ax_roc = fig.add_subplot(gs[1, 0:2])
ax_roc.set_facecolor('#1A2E42')
for sp in ax_roc.spines.values(): sp.set_color('#3A5268')
ax_roc.tick_params(colors=WHITE)
ax_roc.yaxis.label.set_color(WHITE); ax_roc.xaxis.label.set_color(WHITE)
ax_roc.title.set_color(WHITE)
cls_colors_list = [GREEN, AMBER, RED]
for i, (cls, col) in enumerate(zip(CLASS_NAMES, cls_colors_list)):
    binary_labels = (all_labels == i).astype(int)
    fpr_i, tpr_i, _ = roc_curve(binary_labels, all_probs_3[:, i])
    auc_i = roc_auc_score(binary_labels, all_probs_3[:, i])
    ax_roc.plot(fpr_i, tpr_i, color=col, linewidth=2.5,
                label=f"{cls} AUC={auc_i:.3f}")
ax_roc.plot([0,1],[0,1], '--', color='#6688AA', linewidth=1)
ax_roc.set_xlabel("FPR"); ax_roc.set_ylabel("TPR")
ax_roc.set_title("Per-Class ROC Curves (OvR)", fontweight='bold')
ax_roc.legend(facecolor='#1A2E42', labelcolor=WHITE, edgecolor='#3A5268', fontsize=8)
ax_roc.grid(color='#2A4060', linewidth=0.5)

# Confusion matrix
ax_cm = fig.add_subplot(gs[1, 2:4])
ax_cm.set_facecolor('#1A2E42')
cm_dark_cmap = LinearSegmentedColormap.from_list('dark', ['#1A2E42', '#185FA5'])
sns.heatmap(cm, annot=True, fmt='d', cmap=cm_dark_cmap, ax=ax_cm,
            linewidths=1.5, linecolor='#0D1B2A', cbar=False,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            annot_kws={"size":13, "color":"white", "weight":"bold"})
ax_cm.set_title("3-Class Confusion Matrix", fontweight='bold', color=WHITE)
ax_cm.set_xlabel("Predicted", color=WHITE); ax_cm.set_ylabel("True", color=WHITE)
ax_cm.tick_params(colors=WHITE)

# Per-class accuracy bar
ax_bar = fig.add_subplot(gs[2, 0:2])
ax_bar.set_facecolor('#1A2E42')
for sp in ax_bar.spines.values(): sp.set_color('#3A5268')
ax_bar.tick_params(colors=WHITE)
ax_bar.yaxis.label.set_color(WHITE); ax_bar.xaxis.label.set_color(WHITE)
ax_bar.title.set_color(WHITE)
per_class_acc = [(all_preds[all_labels==i] == i).mean()
                  if (all_labels==i).sum() > 0 else 0
                  for i in range(3)]
bars = ax_bar.bar(CLASS_NAMES, per_class_acc, color=cls_colors_list,
                   edgecolor='#1A2E42', width=0.5)
for bar, val in zip(bars, per_class_acc):
    ax_bar.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f"{val:.1%}", ha='center', fontweight='bold', color=WHITE)
ax_bar.set_ylim(0, 1.15)
ax_bar.set_title("Per-Class Recall (Sensitivity)", fontweight='bold')
ax_bar.grid(color='#2A4060', linewidth=0.5)

# Key findings
ax_txt = fig.add_subplot(gs[2, 2:4])
ax_txt.set_facecolor('#0D2136')
for sp in ax_txt.spines.values(): sp.set_color('#3A5268')
ax_txt.set_xticks([]); ax_txt.set_yticks([])
lines = [
    "KEY FINDINGS", "",
    "Model  : ResNet-18  Transfer Learning",
    "Output : 3-Class Softmax", "",
    f"  Class 0 — Normal          (No Stroke)",
    f"  Class 1 — Ischemic        (Blocked vessel)",
    f"  Class 2 — Haemorrhagic    (Bleeding)", "",
    f"Macro AUC-ROC  : {macro_auc:.4f}",
    f"Overall Acc    : {overall_acc:.4f}",
    f"Macro F1       : {macro_f1:.4f}", "",
    "Grad-CAM highlights the exact",
    "brain region driving each diagnosis.",
]
for k, line in enumerate(lines):
    col_t = AMBER if k==0 else WHITE
    fw    = 'bold' if k==0 else 'normal'
    ax_txt.text(0.06, 0.97-k*0.068, line, transform=ax_txt.transAxes,
                va='top', ha='left', color=col_t, fontsize=8.5,
                fontweight=fw, fontfamily='monospace')

plt.show()


In [ ]:
import os
import torch

# Save location (must be writable)
SAVE_DIR = "/kaggle/working"
os.makedirs(SAVE_DIR, exist_ok=True)

SAVE_PATH = os.path.join(SAVE_DIR, "resnet18_stroke_3class.pth")

torch.save({
    "model_state_dict": model.state_dict(),
    "class_names": CLASS_NAMES,
    "num_classes": NUM_CLASSES,
    "val_auc": max(history["val_auc"]),
    "val_acc": max(history["val_acc"]),
    "history": history,
}, SAVE_PATH)

print(f"✅ Model saved -> {SAVE_PATH}")
print(f"Output classes : {CLASS_NAMES}")
print(f"Best Val AUC   : {max(history['val_auc']):.4f}")
print(f"Best Val Acc   : {max(history['val_acc']):.4f}")


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import io

def predict_new_image_3class(img_pil):
    tensor = val_transforms(img_pil.convert("RGB")).unsqueeze(0).to(DEVICE)

    model.eval()
    with torch.no_grad():
        output = model(tensor)
        probs  = torch.softmax(output, dim=1)[0].cpu().numpy()   # (3,)
    pred_idx  = probs.argmax()
    pred_cls  = CLASS_NAMES[pred_idx]
    pred_risk = CLASS_TO_RISK[pred_cls]

    img_np  = np.array(img_pil.convert("RGB").resize((224, 224))) / 255.0
    targets = [ClassifierOutputTarget(int(pred_idx))]
    grayscale_cam = cam(input_tensor=tensor, targets=targets)[0]
    overlay = show_cam_on_image(img_np.astype(np.float32), grayscale_cam, use_rgb=True)

    col = [GREEN, AMBER, RED][pred_idx]

    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    fig.suptitle(
        f"Diagnosis: {pred_cls}  ({pred_risk})\n"
        f"Normal: {probs[0]:.1%}  |  Ischemic: {probs[1]:.1%}  |  Haemorrhagic: {probs[2]:.1%}",
        fontsize=12, fontweight='bold', color=col)
    axes[0].imshow(img_np); axes[0].axis('off'); axes[0].set_title("Uploaded MRI")
    axes[1].imshow(overlay); axes[1].axis('off'); axes[1].set_title("Grad-CAM Overlay")
    plt.tight_layout()
    plt.show()

    print(f"Diagnosis   : {pred_cls}")
    print(f"Risk Label  : {pred_risk}")
    print(f"Confidence  : {probs.max():.2%}")
    print(f"  Normal          : {probs[0]:.2%}")
    print(f"  Ischemic        : {probs[1]:.2%}")
    print(f"  Haemorrhagic    : {probs[2]:.2%}")

upload_widget = widgets.FileUpload(accept='image/*', multiple=False)
button        = widgets.Button(description="Run Diagnosis")
output_area   = widgets.Output()

def on_click(b):
    with output_area:
        clear_output()
        if not upload_widget.value:
            print("Upload a brain MRI image first.")
            return
        uploaded_file = upload_widget.value[0] if isinstance(upload_widget.value, tuple) \
                        else list(upload_widget.value.values())[0]
        content = uploaded_file['content'] if isinstance(uploaded_file, dict) else uploaded_file.content
        img_pil = Image.open(io.BytesIO(content))
        predict_new_image_3class(img_pil)

button.on_click(on_click)
display(widgets.VBox([upload_widget, button, output_area]))
